# Kendall's W rank concordance

## Imports and helpers

In [ ]:
RUN_PATH = "artifacts/simulation_runs/run-20260809T143129.zarr"

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").exists() and (parent / "tasks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
RUN_PATH = Path(RUN_PATH).expanduser()
if not RUN_PATH.is_absolute():
    RUN_PATH = PROJECT_ROOT / RUN_PATH
RUN_PATH = RUN_PATH.resolve()
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures" / RUN_PATH.stem
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401 - registers the science style
import xarray as xr
from IPython.display import display
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib_inline.backend_inline import set_matplotlib_formats
from scipy.stats import bootstrap, ecdf, friedmanchisquare, rankdata

from network_creation.core.observability import ObservationSampling
from network_creation.data.choi_2020 import (
    list_choi_2020_configs,
    load_choi_2020_normalized_data,
    run_choi_2020_replay_batch,
)
from network_creation.data.choi_2020_references import (
    build_choi_observers,
    load_reference_parameters,
)
from network_creation.stats.topology_welfare import (
    reference_centered_mean_welfare,
)
from tasks.policies.results import load_policy_results

%matplotlib inline
plt.style.use("science")
set_matplotlib_formats("svg")
extra = 3
plt.rcParams.update({
    "font.size": 14 + extra,
    "axes.titlesize": 17 + extra,
    "axes.labelsize": 15 + extra,
    "xtick.labelsize": 12 + extra,
    "ytick.labelsize": 12 + extra,
    "legend.fontsize": 12 + extra,
    "legend.title_fontsize": 13 + extra,
    "figure.titlesize": 19 + extra,
})

PAYOFF_NAME = "agent_payoff.runs"
RNG_SEED = 20260729
BOOTSTRAP_RESAMPLES = 5_000
KENDALL_NULL_DRAWS = 8
CONFIDENCE_LEVEL = 0.95
GROUP_SIZE_COLORS = ("#1f4e79", "#2e7d32", "#d97706")
FILTER_ABOVE_MAX_RANDOM_WELFARE = False
WELFARE_SELECTION_LABEL = (
    "Welfare above Max ER"
    if FILTER_ABOVE_MAX_RANDOM_WELFARE
    else "All groups"
)
WELFARE_FILTER_FILENAME = (
    "filter-above-max-er"
    if FILTER_ABOVE_MAX_RANDOM_WELFARE
    else "unfiltered"
)

In [ ]:
def build_simulation_dataset_for_agent_count(
    results,
    n_agents,
    *,
    group_parameter_names,
    payoff_name,
):
    """Build the analysis dataset for one simulated agent count."""
    source = results.sel(n_agents=n_agents)
    payoff = (
        source[payoff_name]
        .isel(agent_slot=slice(0, n_agents))
        .rename(agent_slot="agent")
        .transpose("parameter_row", "agent", "episode")
        .rename("payoff")
    )
    group_parameters = (
        source.theta
        .sel(theta_parameter=list(group_parameter_names))
        .to_dataset(dim="theta_parameter")
        .rename_vars(group_parameter_names)
    )
    return xr.merge((payoff, group_parameters)).assign_attrs(
        n_agents=n_agents
    )


def get_experimental_group_key(config):
    """Return the cohort key shared by an experimental group's rounds."""
    return (
        config.source_path.parent.name,
        config.session_id,
        config.group_id,
    )


def build_subject_to_agent_index_map(config, identity_rows_by_source):
    """Map persistent subject IDs to round-local agent indices."""
    rows = identity_rows_by_source[config.source_path]
    selected = rows.loc[
        (rows["session"] == config.session_id)
        & (rows["grp"] == config.group_id)
        & (rows["round"] == config.round_id),
        ["id_subj", "id_grp"],
    ].drop_duplicates()
    if (
        selected.empty
        or selected["id_subj"].duplicated().any()
        or selected["id_grp"].duplicated().any()
    ):
        raise ValueError(f"Invalid subject-to-agent mapping for {config}")
    return {
        int(subject_id): int(agent_id) - 1
        for subject_id, agent_id in selected.itertuples(index=False, name=None)
    }


In [ ]:
def select_groups_above_random_welfare(data: xr.Dataset) -> xr.Dataset:
    """Return xarray rows whose normalized welfare exceeds Max ER."""
    return data.where(
        data["normalized_group_welfare"]
        > data["normalized_max_random_welfare"],
        drop=True,
    )


def compute_group_kendall_w(payoffs, batch_size=512):
    """Return Kendall's W for every group."""
    payoffs = np.asarray(payoffs)
    n_groups, n_agents, n_episodes = payoffs.shape
    result = np.full(n_groups, np.nan, dtype=np.float64)
    denominator = n_episodes * (n_agents - 1)
    for start in range(0, n_groups, batch_size):
        stop = min(start + batch_size, n_groups)
        batch = np.asarray(payoffs[start:stop], dtype=np.float64)
        all_tied = np.all(batch == batch[:, :1, :], axis=(1, 2))
        valid = ~all_tied
        batch_result = np.full(stop - start, np.nan, dtype=np.float64)
        if np.any(valid):
            statistic = friedmanchisquare(
                *np.moveaxis(batch[valid], 1, 0), axis=-1
            ).statistic
            batch_result[valid] = np.asarray(statistic) / denominator
        result[start:stop] = batch_result
    return np.clip(result, 0.0, 1.0)


def sample_permutation_null_kendall_w(
    payoffs, *, rng, n_draws, batch_size=256
):
    """Sample W after independent within-episode identity permutations."""
    payoffs = np.asarray(payoffs)
    n_groups, n_agents, n_episodes = payoffs.shape
    result = np.full(
        (n_draws, n_groups), np.nan, dtype=np.float64
    )
    mean_rank = (n_agents + 1) / 2.0
    expected_rank_sum = n_episodes * mean_rank
    for start in range(0, n_groups, batch_size):
        stop = min(start + batch_size, n_groups)
        ranks = rankdata(payoffs[start:stop], axis=1)
        rank_variability = np.sum(
            (ranks - mean_rank) ** 2, axis=(1, 2)
        )
        valid = rank_variability > 0.0
        if not np.any(valid):
            continue

        valid_ranks = ranks[valid]
        denominator = n_episodes * rank_variability[valid]
        batch_result = result[:, start:stop]
        for draw in range(n_draws):
            permuted_ranks = rng.permuted(valid_ranks, axis=1)
            rank_sums = permuted_ranks.sum(axis=2, dtype=np.float64)
            batch_result[draw, valid] = (
                ((rank_sums - expected_rank_sum) ** 2).sum(
                    axis=1, dtype=np.float64
                )
                / denominator
            )
    return np.clip(result, 0.0, 1.0)


def compute_ecdf_band_half_width(values, rng, n_resamples, confidence):
    """Return a bootstrap uniform ECDF-band half-width."""
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    _, encoded, counts = np.unique(
        values, return_inverse=True, return_counts=True
    )
    observed_cdf = np.cumsum(counts) / counts.sum()

    def maximum_deviation(sample, axis=-1):
        sample = np.moveaxis(sample, axis, -1)
        rows = sample.reshape(-1, sample.shape[-1])
        deviations = np.empty(len(rows), dtype=np.float64)
        for row_index, row in enumerate(rows):
            bootstrap_cdf = np.cumsum(
                np.bincount(row, minlength=len(observed_cdf))
            ) / len(row)
            deviations[row_index] = np.max(
                np.abs(bootstrap_cdf - observed_cdf)
            )
        return deviations.reshape(sample.shape[:-1])

    result = bootstrap(
        (encoded,),
        maximum_deviation,
        vectorized=True,
        n_resamples=n_resamples,
        batch=32,
        confidence_level=confidence,
        alternative="less",
        method="percentile",
        rng=rng,
    )
    return float(result.confidence_interval.high)

In [ ]:
def plot_kendall_ecdfs(
    axis, observed, null, band_half_width, *, color
):
    """Plot observed and permutation-null ECDFs with a confidence band."""
    observed_cdf = ecdf(observed).cdf
    grid = np.unique(np.concatenate((
        [0.0], observed_cdf.quantiles, [1.0]
    )))
    probabilities = observed_cdf.evaluate(grid)
    axis.fill_between(
        grid,
        np.clip(probabilities - band_half_width, 0.0, 1.0),
        np.clip(probabilities + band_half_width, 0.0, 1.0),
        step="post",
        color=color,
        alpha=0.16,
    )
    ecdf(null).cdf.plot(
        ax=axis,
        color=color,
        linestyle="--",
        linewidth=1.6,
    )
    observed_cdf.plot(ax=axis, color=color, linewidth=2.2)


def plot_group_statistic_parameter_space(
    analysis, values_by_size, *, norm, colorbar_label, title
):
    """Plot one group-level statistic across sampled parameter centers."""
    n_agent_sizes = list(values_by_size)
    visibility_values = np.unique(
        analysis[str(n_agent_sizes[0])]
        ["visibility_center"]
        .values
    )
    n_rows = len(n_agent_sizes)
    n_columns = len(visibility_values)
    figure, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(3.4 * n_columns + 2.8, 3.4 * n_rows + 0.8),
        sharex=True,
        sharey=True,
        gridspec_kw={"wspace": 0.06, "hspace": 0.06},
        squeeze=False,
    )
    figure.subplots_adjust(
        left=0.10, right=0.752, bottom=0.11, top=0.86
    )
    horizontal_scale = 13.0 / figure.get_figwidth()

    mapped = None
    for row_index, n_agents in enumerate(n_agent_sizes):
        size_data = analysis[str(n_agents)]
        values = values_by_size[n_agents]
        parameters = size_data.dataset.sel(
            parameter_row=values["parameter_row"]
        )
        for column_index, visibility in enumerate(visibility_values):
            axis = axes[row_index, column_index]
            mask = np.isclose(
                parameters["visibility_center"].values, visibility
            )
            mapped = axis.scatter(
                parameters["outside_optimism_center"].values[mask],
                parameters["exploration_center"].values[mask],
                c=values.values[mask],
                cmap="jet",
                norm=norm,
                s=4,
                linewidths=0,
                rasterized=True,
            )
            axis.set_box_aspect(1)
            axis.grid(False)

    strip_facecolor = "#f2f2f2"
    strip_edgecolor = "#b7b7b7"
    for column_index, visibility in enumerate(visibility_values):
        position = axes[0, column_index].get_position()
        strip = figure.add_axes([
            position.x0,
            position.y1 + 0.012,
            position.width,
            0.04,
        ])
        strip.set_facecolor(strip_facecolor)
        strip.text(
            0.5,
            0.5,
            f"Visibility-radius center {visibility:g}",
            ha="center",
            va="center",
            fontsize=18,
        )
        strip.set_xticks([])
        strip.set_yticks([])
        for spine in strip.spines.values():
            spine.set_color(strip_edgecolor)
            spine.set_linewidth(0.8)

    row_strip_width = 0.03 * horizontal_scale
    row_strip_left = (
        axes[0, 0].get_position().x0 - 0.035 - row_strip_width
    )
    for row_index, n_agents in enumerate(n_agent_sizes):
        position = axes[row_index, 0].get_position()
        strip = figure.add_axes([
            row_strip_left,
            position.y0,
            row_strip_width,
            position.height,
        ])
        strip.set_facecolor(strip_facecolor)
        strip.text(
            0.5,
            0.5,
            f"{int(n_agents)} agents",
            ha="center",
            va="center",
            rotation=90,
            fontsize=16,
        )
        strip.set_xticks([])
        strip.set_yticks([])
        for spine in strip.spines.values():
            spine.set_color(strip_edgecolor)
            spine.set_linewidth(0.8)

    grid_left = axes[0, 0].get_position().x0
    grid_right = axes[0, -1].get_position().x1
    grid_bottom = axes[-1, 0].get_position().y0
    grid_top = axes[0, 0].get_position().y1
    figure.text(
        (grid_left + grid_right) / 2,
        0.055,
        "Outside-link optimism center",
        ha="center",
        fontsize=16,
    )
    figure.text(
        row_strip_left - 0.023 * horizontal_scale,
        (grid_bottom + grid_top) / 2,
        "Choice-temperature center",
        ha="center",
        va="center",
        rotation=90,
        fontsize=16,
    )

    assert mapped is not None
    colorbar_height = 0.64 * (grid_top - grid_bottom)
    colorbar_axis = figure.add_axes([
        grid_right + 0.048 * horizontal_scale,
        (grid_bottom + grid_top - colorbar_height) / 2,
        0.025 * horizontal_scale,
        colorbar_height,
    ])
    colorbar = figure.colorbar(mapped, cax=colorbar_axis)
    colorbar.set_label(colorbar_label)
    colorbar.ax.yaxis.set_label_position("left")
    colorbar.outline.set_edgecolor("black")
    colorbar.outline.set_linewidth(1.0)
    figure.suptitle(title, y=0.98)
    return figure

## Data loading

In [ ]:
loaded = load_policy_results(
    RUN_PATH,
    stats=PAYOFF_NAME,
    normalize_time=True,
)
if loaded.attrs["has_failures"]:
    raise RuntimeError("The analysis rejects stores containing failed rows")
if loaded.attrs["sweep_layout"] != "cartesian":
    raise ValueError("Expected a Cartesian sweep")

policy_proportion = float(loaded.best_response_proportion.item())
terminal_activation_time = float(loaded.step.item())
loaded = loaded.squeeze(
    dim=("best_response_proportion", "step"), drop=True
)
group_parameter_names = {
    "benefit_to_cost_ratio": "benefit_to_cost_ratio",
    "distance_discount": "distance_discount",
    "visibility_distance": "visibility_center",
    "outside_optimism": "outside_optimism_center",
    "exploration": "exploration_center",
}

n_agents_values = np.asarray(loaded.n_agents.values, dtype=np.int64)
datasets_by_path = {
    "/": xr.Dataset(
        attrs={
            **loaded.attrs,
            "policy_proportion": policy_proportion,
            "terminal_activation_time": terminal_activation_time,
        },
    )
}
for n_agents in n_agents_values:
    datasets_by_path[f"/{n_agents}"] = build_simulation_dataset_for_agent_count(
        loaded,
        int(n_agents),
        group_parameter_names=group_parameter_names,
        payoff_name=PAYOFF_NAME,
    )
analysis = xr.DataTree.from_dict(datasets_by_path)
for n_agents in n_agents_values:
    size_data = analysis[str(n_agents)]
    benefit_to_cost_ratio = size_data["benefit_to_cost_ratio"]
    distance_discount = size_data["distance_discount"]
    maximum_welfare = xr.DataArray(
        [
            reference_centered_mean_welfare(
                reference="max",
                n_agents=int(n_agents),
                benefit_to_cost_ratio=float(benefit),
                distance_discount=float(discount),
            )
            for benefit, discount in zip(
                benefit_to_cost_ratio.values,
                distance_discount.values,
                strict=True,
            )
        ],
        dims="parameter_row",
        coords={"parameter_row": size_data["parameter_row"]},
    )
    random_welfare = xr.DataArray(
        [
            reference_centered_mean_welfare(
                reference="random",
                n_agents=int(n_agents),
                benefit_to_cost_ratio=float(benefit),
                distance_discount=float(discount),
            )
            for benefit, discount in zip(
                benefit_to_cost_ratio.values,
                distance_discount.values,
                strict=True,
            )
        ],
        dims="parameter_row",
        coords={"parameter_row": size_data["parameter_row"]},
    )
    payoff_scale = (
        benefit_to_cost_ratio
        * (int(n_agents) - 1)
        * distance_discount
    )
    size_data["normalized_group_welfare"] = (
        size_data["payoff"].astype(np.float64).mean("agent")
        * payoff_scale
        / (maximum_welfare + 1e-8)
    ).mean("episode")
    size_data["normalized_max_random_welfare"] = (
        random_welfare / maximum_welfare
    )
del datasets_by_path, loaded


analysis

In [ ]:
CHOI_DATA_ROOT = PROJECT_ROOT / "data" / "choi_2020" / "raw"
CHOI_PARAMETERS_PATH = PROJECT_ROOT / "data" / "choi_2020" / "parameters.json"

choi_configs = list_choi_2020_configs(CHOI_DATA_ROOT)
choi_parameters_by_size = {
    row.group_size: row
    for row in load_reference_parameters(CHOI_PARAMETERS_PATH)
}
identity_rows_by_source = {
    source_path: pd.read_csv(
        source_path,
        usecols=("session", "grp", "round", "id_subj", "id_grp"),
    )
    for source_path in {config.source_path for config in choi_configs}
}

experimental_payoffs_by_size = {}
for n_agents, parameters in sorted(choi_parameters_by_size.items()):
    size_configs = tuple(sorted(
        config for config in choi_configs if config.group_size == n_agents
    ))
    max_steps = max(
        load_choi_2020_normalized_data(config).max_step
        for config in size_configs
    )
    payoff_observer = build_choi_observers(
        n_agents=n_agents,
        observer_names=("agent_payoff",),
        observation_sampling=ObservationSampling.terminal(),
        aggregations=("runs",),
        benefit_to_cost_ratio=parameters.benefit_to_cost_ratio,
        distance_discount=parameters.distance_discount,
    )["agent_payoff"]
    run_choi_2020_replay_batch(
        configs=size_configs,
        observers=(payoff_observer,),
        max_steps=max_steps,
    )
    terminal_payoff_by_config = {
        config: np.asarray(run, dtype=np.float64)[-1]
        for config, run in zip(
            size_configs, payoff_observer.get().outputs["runs"], strict=True
        )
    }

    configs_by_group = {}
    for config in size_configs:
        configs_by_group.setdefault(
            get_experimental_group_key(config), []
        ).append(config)

    payoffs_by_group = {}
    for group_key, group_configs in sorted(configs_by_group.items()):
        group_configs = tuple(sorted(group_configs, key=lambda row: row.round_id))
        subject_maps = [
            build_subject_to_agent_index_map(config, identity_rows_by_source)
            for config in group_configs
        ]
        payoffs = xr.concat(
            [
                xr.DataArray(
                    terminal_payoff_by_config[config][list(mapping.values())],
                    dims="subject",
                    coords={"subject": list(mapping)},
                ).expand_dims(round=[config.round_id])
                for config, mapping in zip(
                    group_configs, subject_maps, strict=True
                )
            ],
            dim="round",
            join="inner",
        ).transpose("subject", "round").sortby("subject")
        if payoffs.sizes["subject"] < 3:
            raise ValueError(f"Too few matched subjects for {group_key}")
        payoffs_by_group[group_key] = payoffs
    experimental_payoffs_by_size[n_agents] = payoffs_by_group

## Simulated data

Solid ECDFs show model-simulated Kendall's W, dashed curves show the identity-permutation null, and shading shows a separate uniform group-bootstrap band for each group size.

In [ ]:
kendall_by_size = {}
kendall_null_by_size = {}
kendall_null_rng = np.random.default_rng(RNG_SEED + 4)
for n_agents in sorted(n_agents_values):
    size_data = analysis[str(n_agents)]
    selected = size_data.dataset
    if FILTER_ABOVE_MAX_RANDOM_WELFARE:
        selected = select_groups_above_random_welfare(selected)
    payoffs = selected["payoff"].values
    kendall_w = xr.DataArray(
        compute_group_kendall_w(payoffs),
        dims="parameter_row",
        coords={"parameter_row": selected["parameter_row"].values},
    ).dropna("parameter_row")
    kendall_by_size[int(n_agents)] = kendall_w
    null_w = sample_permutation_null_kendall_w(
        payoffs,
        rng=kendall_null_rng,
        n_draws=KENDALL_NULL_DRAWS,
    )
    kendall_null_by_size[int(n_agents)] = null_w[np.isfinite(null_w)]

kendall_band_rng = np.random.default_rng(RNG_SEED + 3)
kendall_band_half_width_by_size = {
    n_agents: compute_ecdf_band_half_width(
        values.values,
        kendall_band_rng,
        BOOTSTRAP_RESAMPLES,
        CONFIDENCE_LEVEL,
    )
    for n_agents, values in kendall_by_size.items()
}

### Observed groups above the permutation null

The threshold is the 99th percentile of the pooled conditional permutation null for each group size. Shares use a strict exceedance and include only groups with finite observed Kendall's W.

In [ ]:
KENDALL_NULL_THRESHOLD_QUANTILE = 0.99

kendall_null_exceedance_rows = []
for n_agents, observed_values in kendall_by_size.items():
    observed = observed_values.values
    null = kendall_null_by_size[n_agents]
    null_threshold = float(np.quantile(
        null, KENDALL_NULL_THRESHOLD_QUANTILE
    ))
    above_threshold = observed > null_threshold
    kendall_null_exceedance_rows.append({
        "n_agents": n_agents,
        "observed_groups": observed.size,
        "null_samples": null.size,
        "null_quantile": KENDALL_NULL_THRESHOLD_QUANTILE,
        "null_w_threshold": null_threshold,
        "groups_above_threshold": int(above_threshold.sum()),
        "share_above_threshold": float(above_threshold.mean()),
    })

kendall_null_exceedance_summary = pd.DataFrame(
    kendall_null_exceedance_rows
)
display(kendall_null_exceedance_summary.style.format({
    "null_quantile": "{:.0%}",
    "null_w_threshold": "{:.4f}",
    "share_above_threshold": "{:.2%}",
}))

In [ ]:
size_colors = dict(zip(
    kendall_by_size,
    GROUP_SIZE_COLORS,
))
figure, axis = plt.subplots(figsize=(6.5, 6.5), layout="constrained")
for n_agents, values in kendall_by_size.items():
    plot_kendall_ecdfs(
        axis,
        values.values,
        kendall_null_by_size[n_agents],
        kendall_band_half_width_by_size[n_agents],
        color=size_colors[n_agents],
    )

axis.set(
    title="",#f"Identity-linked rank persistence\n{WELFARE_SELECTION_LABEL}",
    xlabel="Kendall's W",
    ylabel="Cumulative proportion of configurations",
    xlim=(0.0, 1.0),
    ylim=(0.0, 1.0),
)
axis.set_box_aspect(1)
axis.minorticks_off()
axis.grid(alpha=0.25)
group_size_handles = [
    Line2D(
        [], [], color=color, linewidth=2.2, label=rf"$n={n_agents}$"
    )
    for n_agents, color in size_colors.items()
]
curve_handles = [
    Line2D([], [], color="black", linewidth=2.2, label="Heterogeneous simulations"),
    Line2D(
        [],
        [],
        color="black",
        linestyle="--",
        linewidth=1.6,
        label="Identity-permutation null",
    ),
    Patch(
        facecolor="black",
        edgecolor="none",
        alpha=0.16,
        label=rf"{CONFIDENCE_LEVEL * 100:.0f}\% simultaneous bootstrap band",
    ),
]
group_size_legend = axis.legend(
    handles=group_size_handles,
    title="Group size",
    loc="upper right",
    frameon=False,
)
axis.add_artist(group_size_legend)
axis.legend(
    handles=curve_handles, title="Curve", loc="lower right", frameon=False
)
figure.savefig(
    FIGURE_DIR / f"kendall_w_ecdf__{WELFARE_FILTER_FILENAME}.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
kendall_norm = Normalize(
    vmin=0.0,
    vmax=1.0,
)
figure = plot_group_statistic_parameter_space(
    analysis,
    kendall_by_size,
    norm=kendall_norm,
    colorbar_label="Kendall's W",
    title="",
)
figure.savefig(
    FIGURE_DIR / f"kendall_w_parameter_space__{WELFARE_FILTER_FILENAME}.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Experimental data

Terminal payoffs are aligned by persistent subject identity across rounds. Each panel shows observed Kendall's W, its permutation null, and its own uniform group-bootstrap band.

In [ ]:
experimental_kendall_null_rng = np.random.default_rng(RNG_SEED + 6)
experimental_kendall_null_by_size = {
    n_agents: [] for n_agents in experimental_payoffs_by_size
}
experimental_kendall_rows = []
for n_agents, payoffs_by_group in experimental_payoffs_by_size.items():
    for group_number, (group_key, payoffs) in enumerate(
        sorted(payoffs_by_group.items()), start=1
    ):
        kendall_w = float(
            compute_group_kendall_w(payoffs.values[None, :, :])[0]
        )
        null_values = sample_permutation_null_kendall_w(
            payoffs.values[None, :, :],
            rng=experimental_kendall_null_rng,
            n_draws=BOOTSTRAP_RESAMPLES,
        )[:, 0]
        experimental_kendall_null_by_size[n_agents].append(
            null_values[np.isfinite(null_values)]
        )
        source_name, session_id, source_group_id = group_key
        experimental_kendall_rows.append({
            "n_agents": n_agents,
            "group": group_number,
            "group_key": (
                f"{source_name}:session={session_id}:group={source_group_id}"
            ),
            "rounds": payoffs.sizes["round"],
            "matched_subjects": payoffs.sizes["subject"],
            "kendall_w": kendall_w,
        })

experimental_kendall_null_by_size = {
    n_agents: np.concatenate(values)
    for n_agents, values in experimental_kendall_null_by_size.items()
}

experimental_kendall_summary = pd.DataFrame(experimental_kendall_rows)
display(experimental_kendall_summary.style.format({
    "kendall_w": "{:.4f}",
}))

In [ ]:
experimental_size_colors = dict(zip(
    sorted(experimental_payoffs_by_size),
    GROUP_SIZE_COLORS,
))
grouped_rows = list(experimental_kendall_summary.groupby(
    "n_agents", sort=True
))
experimental_kendall_band_rng = np.random.default_rng(RNG_SEED + 7)
experimental_kendall_band_half_width_by_size = {
    int(n_agents): compute_ecdf_band_half_width(
        rows["kendall_w"].to_numpy(),
        experimental_kendall_band_rng,
        BOOTSTRAP_RESAMPLES,
        CONFIDENCE_LEVEL,
    )
    for n_agents, rows in grouped_rows
}
figure, axes = plt.subplots(
    1,
    len(grouped_rows),
    figsize=(15.5, 6.5),
    sharex=True,
    sharey=True,
    layout="constrained",
)
for axis, (n_agents, rows) in zip(axes, grouped_rows, strict=True):
    n_agents = int(n_agents)
    color = experimental_size_colors[n_agents]
    observed = rows["kendall_w"].to_numpy()
    plot_kendall_ecdfs(
        axis,
        observed,
        experimental_kendall_null_by_size[n_agents],
        experimental_kendall_band_half_width_by_size[n_agents],
        color=color,
    )
    axis.set(
        title=rf"$n={n_agents}$",
        xlabel="Kendall's W",
        xlim=(0.0, 1.0),
        ylim=(0.0, 1.0),
    )
    axis.set_box_aspect(1)
    axis.minorticks_off()
    axis.grid(alpha=0.25)

axes[0].set_ylabel("Cumulative proportion of configurations")
"""figure.suptitle(
    "Experimental identity-linked rank persistence\n"
    "Choi 2020 terminal states"
)"""
experimental_group_size_handles = [
    Line2D(
        [], [], color=color, linewidth=2.2, label=rf"$n={n_agents}$"
    )
    for n_agents, color in experimental_size_colors.items()
]
curve_handles = [
    Line2D([], [], color="black", linewidth=2.2, label="Observed"),
    Line2D(
        [],
        [],
        color="black",
        linestyle="--",
        linewidth=1.6,
        label="Permutation null",
    ),
    Patch(
        facecolor="black",
        edgecolor="none",
        alpha=0.16,
        label=rf"{CONFIDENCE_LEVEL * 100:.0f}\% bootstrap band",
    ),
]
experimental_legend_handles = [
    handle
    for pair in zip(experimental_group_size_handles, curve_handles, strict=True)
    for handle in pair
]
figure.legend(
    handles=experimental_legend_handles,
    title="Group-size hue (top) and curve style (bottom)",
    loc="outside lower center",
    ncols=len(experimental_group_size_handles),
    frameon=False,
)
figure.savefig(
    FIGURE_DIR / "choi_2020_kendall_w_ecdf.pdf",
    format="pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()